# SWAP-Stress: Model Validation

This notebook evaluates the `direct_rf_9km_global_pruned` Random Forest model
against held-out test data.

1. Load test-set predictions and overall metrics
2. Scatter plots: predicted vs observed by source
3. Residual diagnostics
4. Reconstruct holdout set with coordinates (guarded)
5. Spatial RMSE map (guarded)
6. Temporal evaluation at well-sampled sites (guarded)


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, root_mean_squared_error

try:
    import geopandas as gpd
except Exception:
    gpd = None

# ------------------------------------------------------------------
# Config
# ------------------------------------------------------------------
DATA_ROOT = os.environ.get("SWAPSTRESS_DATA_ROOT", "/nas/soils")
DATA_ROOT = str(Path(DATA_ROOT).expanduser())

MODEL_DIR = os.path.join(DATA_ROOT, "swapstress", "models",
                          "direct_rf_9km_global_pruned")

US_STATES_SHP = "/nas/boundaries/us_states_tiger_wgs.shp"
OUT_DIR = Path("notebooks/_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Guard flags
RUN_RECONSTRUCT = False  # True to merge test split with spatial coords

print("MODEL_DIR:", MODEL_DIR)


In [ ]:
# Load predictions
pred_path = os.path.join(MODEL_DIR, "predictions.parquet")
pred_df = pd.read_parquet(pred_path)
print(f"Predictions shape: {pred_df.shape}")
print(f"Columns: {list(pred_df.columns)}")
print(pred_df.head())

# Load model results
results_path = os.path.join(MODEL_DIR, "direct_model_results.json")
with open(results_path) as f:
    results = json.load(f)

print("\n=== Overall metrics ===")
ov = results.get("overall", results.get("metrics", {}))
for k, v in ov.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

print("\n=== Per-source metrics ===")
src_metrics = results.get("source_metrics", {})
rows = []
for src, m in src_metrics.items():
    rows.append({"source": src, **m})
if rows:
    src_df = pd.DataFrame(rows).set_index("source")
    print(src_df.to_string())
else:
    print("(no per-source metrics in results JSON)")


## 2) Scatter: Predicted vs Observed by Source

One panel per training source.  The diagonal is the 1:1 line.  Each panel is
annotated with R² and RMSE for that source.


In [ ]:
sources = sorted(pred_df["source"].unique())
print(f"Sources ({len(sources)}): {sources}")

ncols = min(3, len(sources))
nrows = (len(sources) + ncols - 1) // ncols

source_colors = {
    "gshp":      "#4e79a7",
    "mt_mesonet": "#f28e2b",
    "reesh":     "#e15759",
    "ncss":      "#76b7b2",
    "lacadian":  "#59a14f",
}
default_color = "#9c755f"

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(5 * ncols, 4.5 * nrows),
                          squeeze=False)

all_vals = pd.concat([pred_df["observed"], pred_df["predicted"]])
lo, hi = np.percentile(all_vals.dropna(), [1, 99])

for idx, src in enumerate(sources):
    row_i, col_i = divmod(idx, ncols)
    ax = axes[row_i][col_i]

    sub = pred_df[pred_df["source"] == src].dropna(
        subset=["observed", "predicted"]
    )
    color = source_colors.get(src, default_color)

    ax.scatter(sub["observed"], sub["predicted"],
                alpha=0.3, s=6, color=color, rasterized=True)
    ax.plot([lo, hi], [lo, hi], "k--", lw=0.8, alpha=0.6)

    r2   = r2_score(sub["observed"], sub["predicted"])
    rmse = root_mean_squared_error(sub["observed"], sub["predicted"])
    ax.text(0.04, 0.94, f"R²={r2:.2f}  RMSE={rmse:.3f}",
             transform=ax.transAxes, fontsize=8, va="top")

    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel("Observed log10 suction (cm)", fontsize=8)
    ax.set_ylabel("Predicted", fontsize=8)
    ax.set_title(f"{src}  (n={len(sub):,})", fontsize=10)
    ax.tick_params(labelsize=7)

# Hide unused panels
for idx in range(len(sources), nrows * ncols):
    row_i, col_i = divmod(idx, ncols)
    axes[row_i][col_i].set_visible(False)

fig.suptitle("Predicted vs observed log10 suction — by source",
              fontsize=13)
fig.tight_layout()
out = OUT_DIR / "08_scatter_by_source.png"
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", out)


## 3) Residual Diagnostics

Residual = predicted − observed.  We check for heteroscedasticity (residuals
vs observed) and compare residual distributions across sources.

Expected: NCSS is the weakest source (R²≈0.29); LaCADIAN also has high scatter
(R²≈0.09).  ReESH / MT Mesonet should be the strongest.


In [ ]:
pred_df["residual"] = pred_df["predicted"] - pred_df["observed"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: residuals vs observed ──────────────────────────────────────────
for src in sources:
    sub = pred_df[pred_df["source"] == src].dropna(
        subset=["observed", "residual"]
    )
    color = source_colors.get(src, default_color)
    ax1.scatter(sub["observed"], sub["residual"],
                 alpha=0.2, s=4, color=color, label=src, rasterized=True)

ax1.axhline(0, color="k", lw=0.8, linestyle="--")
ax1.set_xlabel("Observed log10 suction (cm)")
ax1.set_ylabel("Residual (predicted − observed)")
ax1.set_title("Residuals vs observed")
ax1.legend(markerscale=3, fontsize=8)

# ── Right: residual distribution per source (KDE) ────────────────────────
try:
    from scipy.stats import gaussian_kde
    for src in sources:
        sub = pred_df[pred_df["source"] == src]["residual"].dropna()
        color = source_colors.get(src, default_color)
        kde = gaussian_kde(sub)
        x = np.linspace(sub.min(), sub.max(), 300)
        ax2.plot(x, kde(x), color=color, lw=1.5, label=src)
except ImportError:
    for src in sources:
        sub = pred_df[pred_df["source"] == src]["residual"].dropna()
        color = source_colors.get(src, default_color)
        ax2.hist(sub, bins=60, density=True, alpha=0.5,
                  color=color, label=src)

ax2.axvline(0, color="k", lw=0.8, linestyle="--")
ax2.set_xlabel("Residual (predicted − observed)")
ax2.set_ylabel("Density")
ax2.set_title("Residual distribution per source")
ax2.legend(fontsize=8)

fig.suptitle("Residual diagnostics", fontsize=13)
fig.tight_layout()
out = OUT_DIR / "08_residuals.png"
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", out)

# Summary stats
print("\nPer-source residual stats:")
print(
    pred_df.groupby("source")["residual"]
    .agg(["mean", "std", lambda s: np.sqrt((s**2).mean())])
    .rename(columns={"<lambda_0>": "RMSE"})
    .round(4)
)


## 4) Reconstruct Holdout Set (guarded: `RUN_RECONSTRUCT = False`)

The `predictions.parquet` file has no lat/lon/date — it is aligned row-for-row
with the deterministic test split.  Re-running the same split recovers spatial
and temporal coordinates for Sections 5 and 6.

Set `RUN_RECONSTRUCT = True` to enable Sections 5 and 6.


In [ ]:
test_df = None
merged_df = None

if RUN_RECONSTRUCT:
    from map.learning.decision_tree.train_direct import (
        create_site_split,
        apply_site_split,
    )

    obs_path = os.path.join(
        DATA_ROOT, "swapstress", "training",
        "obs_level_training_9km_global.parquet",
    )
    obs = pd.read_parquet(obs_path)
    print(f"Obs table shape: {obs.shape}")

    train_groups, test_groups = create_site_split(
        obs, random_state=42, resolution_m=250
    )
    _, test_df = apply_site_split(
        obs, train_groups, test_groups, resolution_m=250
    )
    print(f"Test set shape:  {test_df.shape}")

    # Align on row order — same deterministic split
    assert len(test_df) == len(pred_df), (
        f"Row count mismatch: test_df={len(test_df)}, "
        f"pred_df={len(pred_df)}"
    )
    merged_df = test_df.reset_index(drop=True).join(
        pred_df[["observed", "predicted", "source"]].reset_index(drop=True)
    )
    merged_df["residual"] = merged_df["predicted"] - merged_df["observed"]
    print("Merged columns:", list(merged_df.columns))
else:
    print("RUN_RECONSTRUCT=False — skipping holdout reconstruction")


## 5) Spatial RMSE Map (guarded — requires Section 4)

Group test-set predictions by spatial group, compute per-site RMSE, and plot
as a scatter map coloured by RMSE.  Weak performance in arid or heavily-sampled
regions indicates domain gaps.


In [ ]:
if RUN_RECONSTRUCT and merged_df is not None:
    from map.learning.decision_tree.train_direct import assign_spatial_group

    merged_df["spatial_group"] = assign_spatial_group(
        merged_df, resolution_m=250
    )

    # Per-site RMSE
    site_rmse = (
        merged_df.groupby("spatial_group")
        .apply(
            lambda g: pd.Series({
                "rmse": np.sqrt(((g["predicted"] - g["observed"]) ** 2).mean()),
                "n":    len(g),
                "lat":  g["lat"].mean(),
                "lon":  g["lon"].mean(),
            })
        )
        .reset_index()
    )
    site_rmse = site_rmse[site_rmse["n"] >= 3].copy()
    print(f"Sites with ≥3 obs: {len(site_rmse):,}")
    print(f"RMSE range: {site_rmse['rmse'].min():.3f} – "
          f"{site_rmse['rmse'].max():.3f}")

    fig, ax = plt.subplots(figsize=(14, 7))
    sc = ax.scatter(
        site_rmse["lon"], site_rmse["lat"],
        c=site_rmse["rmse"], cmap="RdYlGn_r",
        vmin=np.percentile(site_rmse["rmse"], 2),
        vmax=np.percentile(site_rmse["rmse"], 98),
        s=20, alpha=0.7, edgecolors="none",
    )
    cb = fig.colorbar(sc, ax=ax, shrink=0.7, pad=0.02)
    cb.set_label("RMSE (log10 suction cm)")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Per-site RMSE on test set")
    fig.tight_layout()
    out = OUT_DIR / "08_spatial_rmse.png"
    fig.savefig(out, dpi=200)
    plt.show()
    print("Saved:", out)
else:
    print("Section 4 not run — skipping spatial RMSE map")


## 6) Temporal Evaluation at Well-Sampled Sites (guarded — requires Section 4)

Find the top-3 sites by observation count in the test set (ReESH or MT Mesonet
sites typically dominate).  Plot predicted vs observed log10 suction as a time
series at each site.


In [ ]:
if RUN_RECONSTRUCT and merged_df is not None:
    # Sites must have a date column to plot time series
    date_col = next(
        (c for c in merged_df.columns if "date" in c.lower()), None
    )
    if date_col is None:
        print("No date column found — cannot plot temporal series")
    else:
        merged_df[date_col] = pd.to_datetime(merged_df[date_col])

        top_sites = (
            merged_df.groupby("spatial_group")["observed"]
            .count()
            .nlargest(3)
            .index.tolist()
        )
        print("Top-3 sites by test-set obs:", top_sites)

        fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

        for ax, site in zip(axes, top_sites):
            sub = merged_df[merged_df["spatial_group"] == site].sort_values(date_col)
            src = sub["source"].iloc[0] if "source" in sub else "?"
            ax.plot(sub[date_col], sub["observed"],
                     "o-", ms=3, lw=0.8, color="steelblue", label="Observed")
            ax.plot(sub[date_col], sub["predicted"],
                     "s--", ms=3, lw=0.8, color="tomato",    label="Predicted")
            r2   = r2_score(sub["observed"], sub["predicted"])
            rmse = root_mean_squared_error(sub["observed"], sub["predicted"])
            ax.set_title(f"{site}  [{src}]  R²={r2:.2f}  RMSE={rmse:.3f}",
                          fontsize=10)
            ax.set_ylabel("log10 suction (cm)", fontsize=8)
            ax.legend(fontsize=8)
            ax.tick_params(labelsize=7)

        fig.suptitle("Temporal evaluation at top-3 test-set sites", fontsize=13)
        fig.tight_layout()
        out = OUT_DIR / "08_temporal_sites.png"
        fig.savefig(out, dpi=200)
        plt.show()
        print("Saved:", out)
else:
    print("Section 4 not run — skipping temporal site evaluation")
